# Top-down uncertainty from QC and PT data (Nordtest)

**Question:** what is the expanded uncertainty of a routine copper result, using only
data the lab already collects: its control-sample results and its proficiency-testing
(PT) rounds?

This is the Nordtest TR 537 approach. Instead of modelling every step of the method
(the GUM bottom-up route, in its own notebook), it takes two numbers that already
contain the effects of all those steps:

- **u(R<sub>w</sub>)**, within-laboratory reproducibility, from a control sample run over a long period
- **u(bias)**, method and laboratory bias, from PT rounds (or a certified reference material)

> **Data and assumptions.** All data are synthetic, made by `scripts/make_data.py`:
> copper (Cu) in water by ICP-OES, in µg/L. The fictional lab has a true bias of
> +1.2 % and a within-lab reproducibility of 3 % RSD; the notebook does not know this
> and has to estimate both. The control sample is assumed to go through the **whole**
> procedure, including sample preparation. No real laboratory data are used.

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from labqc.proficiency import en_number
from labqc.topdown import combine_topdown, u_bias_pt, u_rw

DATA = Path("..") / "data" / "synthetic"
if not DATA.exists():  # run from the repository root
    DATA = Path("data") / "synthetic"

qc = pd.read_csv(DATA / "qc.csv", parse_dates=["date"])
pt = pd.read_csv(DATA / "pt.csv", parse_dates=["date"])

# One quiet chart style for the handbook: one series colour, grey references.
BLUE, GREY, INK = "#2a78d6", "#8a8984", "#52514e"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb",
    "axes.facecolor": "#fcfcfb",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": GREY,
    "axes.labelcolor": INK,
    "xtick.color": INK,
    "ytick.color": INK,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "axes.axisbelow": True,
    "grid.color": "#e6e5e0",
    "grid.linewidth": 0.8,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
})

print(f"QC: {len(qc)} runs, {qc['date'].min():%d %b %Y} to {qc['date'].max():%d %b %Y}")
lo, hi = pt["x_pt_ug_L"].min(), pt["x_pt_ug_L"].max()
print(f"PT: {len(pt)} rounds, {pt['date'].min():%b %Y} to {pt['date'].max():%b %Y}, "
      f"assigned values {lo:.0f} to {hi:.0f} µg/L")

## The model

The two components are independent, so they add as variances:

$$
u_c = \sqrt{u(R_w)^2 + u(\text{bias})^2}, \qquad U = k\,u_c \quad (k = 2)
$$

Everything here is **relative, in percent**. The PT rounds span 20 to 150 µg/L, and a
relative uncertainty is the natural way to carry one estimate across that range.
Every term must be in the same form before combining: never mix % and µg/L.

## Step 1: within-lab reproducibility, u(R<sub>w</sub>)

The control sample was run 60 times over about five months, across calibrations,
reagent lots and days. Its relative standard deviation is u(R<sub>w</sub>):

$$
u(R_w) = \frac{s_\text{control}}{\bar{x}_\text{control}} \times 100\,\%
$$

If the control did *not* go through sample preparation (a standard solution, say),
the repeatability of duplicate real samples would be added: `u_rw(..., s_r=...)`.

In [ ]:
u_rw_pct = u_rw(qc["cu_ug_L"], relative=True)

mean, s = qc["cu_ug_L"].mean(), qc["cu_ug_L"].std(ddof=1)
print(f"Control mean {mean:.2f} µg/L, s = {s:.2f} µg/L")
print(f"u(Rw) = {u_rw_pct:.2f} %")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(qc["date"], qc["cu_ug_L"], color=BLUE, lw=1, marker="o", ms=4)
for level, style, label in [
    (mean, "-", f"mean {mean:.2f}"),
    (mean + 2 * s, ":", "mean + 2s"),
    (mean - 2 * s, ":", "mean − 2s"),
]:
    ax.axhline(level, color=GREY, ls=style, lw=1)
    ax.annotate(label, (1.0, level), xycoords=("axes fraction", "data"),
                xytext=(4, 0), textcoords="offset points",
                va="center", fontsize=9, color=INK)
ax.set_ylabel("Cu (µg/L)")
ax.set_title("Control sample, target 50.0 µg/L: 60 runs")
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.tight_layout()
plt.show()

The results scatter evenly around a mean slightly above the 50.0 µg/L target, with no
trend or shift, so the whole series can be used. (Had there been a step change, from a
new instrument or method change, only the data after it would describe the method as
it runs today. Chapter 4 covers how to spot that.)

## Step 2: bias from proficiency testing, u(bias)

For each PT round *i*, the relative bias is the lab's result against the assigned value:

$$
b_i = \frac{x_i - X_i}{X_i} \times 100\,\%, \qquad
\text{RMS}_\text{bias} = \sqrt{\tfrac{1}{n}\textstyle\sum_{i=1}^{n} b_i^2}
$$

The assigned values are themselves uncertain, so their mean relative standard
uncertainty, u(C<sub>ref</sub>), is added:

$$
u(\text{bias}) = \sqrt{\text{RMS}_\text{bias}^2 + u(C_\text{ref})^2}
$$

Nordtest asks for at least six rounds; here there are twelve.

In [ ]:
bias = u_bias_pt(
    pt["lab_result_ug_L"], pt["x_pt_ug_L"], pt["u_x_pt_ug_L"], relative=True
)

rounds = bias.table.assign(round=pt["round"], date=pt["date"].dt.strftime("%b %Y"))
display(
    rounds[["round", "date", "result", "assigned", "bias", "u_ref"]]
    .rename(columns={"result": "lab (µg/L)", "assigned": "X (µg/L)",
                     "bias": "bias b_i (%)", "u_ref": "u(X) (%)"})
    .round(2).set_index("round")
)
print(f"RMS bias = {bias.bias:.2f} %, u(Cref) = {bias.u_ref:.2f} %, "
      f"u(bias) = {bias.u_bias:.2f} %")

In [ ]:
mean_b = bias.table["bias"].mean()

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.bar(pt["round"], bias.table["bias"], color=BLUE, width=0.6)
ax.axhline(0, color=INK, lw=0.8)
ax.axhline(mean_b, color=GREY, ls="--", lw=1)
ax.annotate(f"mean bias {mean_b:+.2f} %", (1.0, mean_b),
            xycoords=("axes fraction", "data"), xytext=(4, 0),
            textcoords="offset points", va="center", fontsize=9, color=INK)
ax.set_xticks(pt["round"])
ax.set_xlabel("PT round")
ax.set_ylabel("Relative bias (%)")
ax.set_title(f"Bias in 12 PT rounds: RMS {bias.bias:.2f} %")
plt.tight_layout()
plt.show()

The bias swings between about −3 % and +6 % from round to round, around a mean of
only +1.3 %. Most of the RMS is therefore **round-to-round scatter**, not a fixed
offset. Keep that in mind for Step 5.

## Step 3: combine

In [ ]:
result = combine_topdown(u_rw_pct, bias, k=2)
print(result.summary())

display(
    result.table.assign(u=result.table["u"].round(2),
                        share=(100 * result.table["share_of_variance"]).round(0))
    .drop(columns="share_of_variance")
    .rename(columns={"u": "u (%)", "share": "share of u_c² (%)"})
    .set_index("term")
)

The bias term carries more than half of the variance. That is typical of the
top-down approach: the lab's own precision is usually the smaller part, and the
quickest way to a smaller U is better information about bias, not a tighter
control chart.

### Check the arithmetic by hand

The same numbers with numpy only, so nothing is hidden inside `labqc`:

In [ ]:
x = qc["cu_ug_L"].to_numpy()
by_hand_rw = 100 * x.std(ddof=1) / x.mean()

b = 100 * (pt["lab_result_ug_L"] - pt["x_pt_ug_L"]) / pt["x_pt_ug_L"]
u_cref = (100 * pt["u_x_pt_ug_L"] / pt["x_pt_ug_L"]).mean()
by_hand_bias = np.sqrt(np.mean(b**2) + u_cref**2)

by_hand_U = 2 * np.sqrt(by_hand_rw**2 + by_hand_bias**2)
print(f"u(Rw) {by_hand_rw:.3f} %, u(bias) {by_hand_bias:.3f} %, U {by_hand_U:.3f} %")
assert np.isclose(by_hand_U, result.expanded), "labqc and the hand sum disagree"
print("labqc agrees with the hand calculation.")

## Step 4: report a result

For a sample result of 48.2 µg/L, the relative U becomes an absolute one:

In [ ]:
y = 48.2
U_abs = result.expanded / 100 * y
print(f"Cu = {y:.1f} ± {U_abs:.1f} µg/L (k = 2, about 95 %)")
print(f"Interval: {y - U_abs:.1f} to {y + U_abs:.1f} µg/L")

Chapter 1 used this same sample with U = 3.4 µg/L, the **7 %** the fictional lab
currently quotes. The top-down estimate is larger, about **9 %**. Either way, the
interval contains the hypothetical 50 µg/L limit; the decision rule (chapter 4)
decides what the report says.

## Step 5: is the estimate believable?

Two checks before using the number.

**Does it agree with the PT results?** The E<sub>n</sub> number compares each PT
result with the assigned value, using both expanded uncertainties:

$$
E_n = \frac{x_i - X_i}{\sqrt{U_\text{lab}^2 + U_\text{ref}^2}}, \qquad |E_n| \le 1 \text{ is satisfactory}
$$

If the lab's U were too small, some rounds would show |E<sub>n</sub>| > 1.

In [ ]:
U_ref = 2 * pt["u_x_pt_ug_L"]  # expanded uncertainty of the assigned values, k = 2
for label, U_rel in [("top-down U", result.expanded), ("the lab's 7 %", 7.0)]:
    U_lab = U_rel / 100 * pt["lab_result_ug_L"]
    en = [en_number(x, X, Ul, Ur) for x, X, Ul, Ur
          in zip(pt["lab_result_ug_L"], pt["x_pt_ug_L"], U_lab, U_ref)]
    print(f"{label:14s} ({U_rel:.1f} %): largest |En| = {max(map(abs, en)):.2f}")

Both are comfortably below 1, so neither U is contradicted by the PT history. That
check alone cannot choose between them.

**How much does the RMS choice matter?** Nordtest uses the RMS of the PT biases.
Each round's bias, though, also contains that round's random error, which u(R<sub>w</sub>)
already counts. An alternative treats the bias as one fixed offset, estimated by the
**mean** bias with its standard uncertainty, as the CRM formula does:

$$
u(\text{bias})_\text{alt} = \sqrt{\bar{b}^2 + \left(\frac{s_b}{\sqrt{n}}\right)^2 + u(C_\text{ref})^2}
$$

In [ ]:
n = len(b)
u_bias_alt = np.sqrt(b.mean() ** 2 + (b.std(ddof=1) / np.sqrt(n)) ** 2 + u_cref**2)
alt = combine_topdown(u_rw_pct, u_bias_alt, k=2, relative=True)

print(f"Nordtest (RMS):   u(bias) = {bias.u_bias:.2f} %, U = {result.expanded:.1f} %")
print(f"Mean-bias route:  u(bias) = {u_bias_alt:.2f} %, U = {alt.expanded:.1f} %")

The alternative gives about 7 %, the lab's current figure; the Nordtest default
gives about 9 %. The difference is the round-to-round scatter seen in Step 2,
counted once (mean-bias route) or twice (RMS route).

The RMS route is deliberately conservative, and it is what Nordtest TR 537
describes. The mean-bias route assumes the bias really is the same at every level
and in every matrix the PT rounds covered. Whichever route the lab takes, it
should state the choice and the reason in its uncertainty procedure.

## What this shows

- With 60 control results and 12 PT rounds, the top-down expanded uncertainty for
  this Cu method is about **9 % (k = 2)**, valid across the 20 to 150 µg/L range the
  data cover.
- Bias, not precision, is the larger component. Better bias information (a matrix
  CRM, more PT rounds) would reduce U more than tighter control.
- The result depends on how PT bias is summarised. That is a method decision to
  document, not a calculation error.

> **From the lab.** *[To write: how a lab you worked in used QC and PT data for
> uncertainty, and how an assessor reacted to a top-down estimate compared with a
> full budget.]*

**Limits of this estimate.** Near the LOQ a relative uncertainty breaks down, because
absolute noise dominates; there the uncertainty is better stated in µg/L. The
estimate covers only the matrices and levels the QC and PT data represent.

## References

- Nordtest TR 537, *Handbook for calculation of measurement uncertainty in
  environmental laboratories* (check the current edition).
- ISO 13528, statistical methods for proficiency testing (E<sub>n</sub> numbers).
- `labqc.topdown`: [API documentation](https://mantas123456.github.io/lab-quality-toolkit/api/topdown/).